In [1]:
# ╔══════════════════════════════════════════════════════════╗
#  CELL 1 — Install · Imports · Environment Report
#  ConvNeXt V2 — Resumed Training (Phase 2 Continuation)
# ╚══════════════════════════════════════════════════════════╝
import subprocess, sys

print("Step 1/3 — Installing timm ...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       '-q', '--no-warn-script-location', 'timm'])
print("  ✓ timm installed")

print("Step 2/3 — Importing libraries ...")
import os
import gc
import random
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
import torchvision.transforms as T
import timm

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    f1_score,
)
import matplotlib.pyplot as plt
from tqdm import tqdm

warnings.filterwarnings('ignore')
print("  ✓ All imports successful")

print("Step 3/3 — Reproducibility & device ...")
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print()
print("=" * 60)
print("  ENVIRONMENT REPORT")
print("=" * 60)
print(f"  Python   : {sys.version.split()[0]}")
print(f"  PyTorch  : {torch.__version__}")
print(f"  timm     : {timm.__version__}")
print(f"  numpy    : {np.__version__}")
print(f"  pandas   : {pd.__version__}")
print(f"  Device   : {DEVICE}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info(0)
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {total/1e9:.1f} GB total | {free/1e9:.2f} GB free")
    print(f"  CUDA     : {torch.version.cuda}")
    print(f"  AMP      : ✓ available")
else:
    print("  ⚠ No GPU detected — training will be very slow")

# ── Verify checkpoint path exists ────────────────────────────
CKPT_PATH = '/kaggle/input/models/omkarphadtare30/convnextv2/other/default/1/best_convnextv2.pth'
print()
print("=" * 60)
print("  CHECKPOINT VERIFICATION")
print("=" * 60)
if os.path.exists(CKPT_PATH):
    size_mb = os.path.getsize(CKPT_PATH) / 1e6
    print(f"  ✓ Checkpoint found  : {CKPT_PATH}")
    print(f"  ✓ File size         : {size_mb:.1f} MB")
else:
    print(f"  ✗ NOT FOUND         : {CKPT_PATH}")
    print("    → Make sure the Kaggle model is attached to this notebook")

print("=" * 60)
print("✓ Cell 1 complete")


Step 1/3 — Installing timm ...
  ✓ timm installed
Step 2/3 — Importing libraries ...
  ✓ All imports successful
Step 3/3 — Reproducibility & device ...

  ENVIRONMENT REPORT
  Python   : 3.12.12
  PyTorch  : 2.9.0+cu126
  timm     : 1.0.24
  numpy    : 2.0.2
  pandas   : 2.3.3
  Device   : cuda
  GPU      : Tesla T4
  VRAM     : 15.6 GB total | 15.53 GB free
  CUDA     : 12.6
  AMP      : ✓ available

  CHECKPOINT VERIFICATION
  ✓ Checkpoint found  : /kaggle/input/models/omkarphadtare30/convnextv2/other/default/1/best_convnextv2.pth
  ✓ File size         : 353.0 MB
✓ Cell 1 complete


In [2]:
# ╔══════════════════════════════════════════════════════════╗
#  CELL 2 — Config · Paths · Load CSVs · Class Weights
#  KEY CHANGES vs last run:
#   - phase2_epochs = 15  (13 new epochs + 2 buffer)
#   - phase2_patience = 8  (don't stop early on fluctuations)
#   - checkpoint_path points to Kaggle model input
# ╚══════════════════════════════════════════════════════════╝

TRAIN_START = time.time()   # global timer for budget guard

# ── Paths ─────────────────────────────────────────────────────
CKPT_PATH  = '/kaggle/input/models/omkarphadtare30/convnextv2/other/default/1/best_convnextv2.pth'
BASE       = Path('/kaggle/input/datasets/omkarphadtare30/deepguard-dataset')
CSV_DIR    = BASE / 'csvs'

# ── Config ────────────────────────────────────────────────────
CFG = dict(
    # Model
    model_name         = 'convnextv2_base',
    num_classes        = 2,
    dropout            = 0.4,
    drop_path_rate     = 0.2,
    img_size           = 224,
    img_mean           = (0.485, 0.456, 0.406),
    img_std            = (0.229, 0.224, 0.225),

    # DataLoader
    num_workers        = 4,

    # Phase 2 (resumed — NO Phase 1)
    phase2_epochs      = 15,          # 13 real + 2 buffer
    phase2_lr_backbone = 5e-6,        # same as last session
    phase2_lr_head     = 5e-5,        # same as last session
    phase2_patience    = 8,           # increased from 4
    phase2_batch       = 32,          # same as last session
    accum_steps        = 2,           # eff. batch = 64

    # Regularisation (identical to last session)
    weight_decay       = 2e-4,
    label_smoothing    = 0.05,
    grad_clip          = 1.0,
    mixup_alpha        = 0.4,
    cutmix_alpha       = 0.4,
    mix_prob           = 0.5,

    # LR scheduler
    rlrop_factor       = 0.5,
    rlrop_patience     = 3,
    rlrop_min_lr       = 1e-8,

    # Time budget — 11.5h, leaves 30 min for testing
    time_budget_sec    = 11.5 * 3600,

    # Output checkpoint
    model_path         = '/kaggle/working/best_convnextv2.pth',
)

# ── Load CSVs ─────────────────────────────────────────────────
print("=" * 60)
print("  LOADING CSVs ...")
print("=" * 60)
train_df = pd.read_csv(CSV_DIR / 'fixed_train.csv')
val_df   = pd.read_csv(CSV_DIR / 'fixed_val.csv')
test_df  = pd.read_csv(CSV_DIR / 'fixed_test.csv')

PATH_COL  = 'imagepath'
LABEL_COL = 'label'
print(f"  Columns  : {list(train_df.columns)}")
print()
print(f"  {'Split':<8} {'Total':>8}  {'Real':>8}  {'Fake':>8}")
print("  " + "-" * 36)
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    r = (df[LABEL_COL] == 0).sum()
    f = (df[LABEL_COL] == 1).sum()
    print(f"  {name:<8} {len(df):>8,}  {r:>8,}  {f:>8,}")

# ── Spot-check 10 paths ───────────────────────────────────────
print()
sample = train_df.sample(10, random_state=42)
broken = [row[PATH_COL] for _, row in sample.iterrows()
          if not os.path.exists(row[PATH_COL])]
if broken:
    print(f"  ⚠ {len(broken)}/10 paths broken!")
    for b in broken[:3]:
        print(f"     {b}")
else:
    print("  ✓ All 10 paths valid")

# ── Class weights ─────────────────────────────────────────────
counts        = train_df[LABEL_COL].value_counts().sort_index().values
class_weights = torch.tensor([1.0 / c for c in counts], dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * 2).to(DEVICE)

print()
print(f"  Class weights → Real: {class_weights[0]:.6f}  Fake: {class_weights[1]:.6f}")

torch.backends.cudnn.benchmark = True

print()
print("=" * 60)
print(f"  SESSION PLAN")
print("=" * 60)
print(f"  Resuming from   : epoch 6 (Val Acc 93.98% | AUC 0.9848)")
print(f"  Running         : 13 more Phase 2 epochs  (epochs 7–19)")
print(f"  Time budget     : {CFG['time_budget_sec']/3600:.1f}h")
print(f"  Batch size      : {CFG['phase2_batch']}  (eff. {CFG['phase2_batch'] * CFG['accum_steps']})")
print(f"  Backbone LR     : {CFG['phase2_lr_backbone']}")
print(f"  Head LR         : {CFG['phase2_lr_head']}")
print(f"  Patience        : {CFG['phase2_patience']} epochs")
print(f"  Output ckpt     : {CFG['model_path']}")
print("=" * 60)
print("✓ Cell 2 complete")


  LOADING CSVs ...
  Columns  : ['imagepath', 'label', 'split']

  Split       Total      Real      Fake
  ------------------------------------
  train     239,828   119,914   119,914
  val        32,374    16,187    16,187
  test       15,982     7,991     7,991

  ✓ All 10 paths valid

  Class weights → Real: 1.000000  Fake: 1.000000

  SESSION PLAN
  Resuming from   : epoch 6 (Val Acc 93.98% | AUC 0.9848)
  Running         : 13 more Phase 2 epochs  (epochs 7–19)
  Time budget     : 11.5h
  Batch size      : 32  (eff. 64)
  Backbone LR     : 5e-06
  Head LR         : 5e-05
  Patience        : 8 epochs
  Output ckpt     : /kaggle/working/best_convnextv2.pth
✓ Cell 2 complete


In [3]:
# ╔══════════════════════════════════════════════════════════╗
#  CELL 3 — Transforms
#  Phase 2 : strong augmentation (full fine-tune)
#  Val/Test : clean deterministic
#  NOTE: TTA transforms defined later in Cell 8 (testing)
# ╚══════════════════════════════════════════════════════════╝

S    = CFG['img_size']   # 224
MEAN = CFG['img_mean']   # (0.485, 0.456, 0.406)
STD  = CFG['img_std']    # (0.229, 0.224, 0.225)

# ── Phase 2 — strong aug, identical to last session ───────────
phase2_transform = T.Compose([
    T.Resize((S + 32, S + 32)),
    T.RandomCrop(S),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3,
                  contrast=0.3,
                  saturation=0.2,
                  hue=0.05),
    T.RandomRotation(degrees=12),
    T.RandomGrayscale(p=0.05),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    T.RandomPerspective(distortion_scale=0.2, p=0.3),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.15, scale=(0.02, 0.10)),
])

# ── Val / Test — clean, deterministic ────────────────────────
val_transform = T.Compose([
    T.Resize((S, S)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# ── Verify transforms on a real image ────────────────────────
sample_path = train_df[PATH_COL].iloc[0]
sample_img  = Image.open(sample_path).convert('RGB')

t2 = phase2_transform(sample_img)
tv = val_transform(sample_img)

print("=" * 60)
print("  TRANSFORM VERIFICATION")
print("=" * 60)
print(f"  Sample image size  : {sample_img.size}")
print(f"  Phase 2 output     : {tuple(t2.shape)}  "
      f"min={t2.min():.3f}  max={t2.max():.3f}")
print(f"  Val output         : {tuple(tv.shape)}  "
      f"min={tv.min():.3f}  max={tv.max():.3f}")
print(f"  Phase 2 steps      : {len(phase2_transform.transforms)}")
print(f"  Val/Test steps     : {len(val_transform.transforms)}")
print()
print("  NOTE: TTA transforms will be defined in Cell 8 after training")
print("=" * 60)
print("✓ Cell 3 complete")


  TRANSFORM VERIFICATION
  Sample image size  : (256, 256)
  Phase 2 output     : (3, 224, 224)  min=-2.118  max=0.879
  Val output         : (3, 224, 224)  min=-2.101  max=0.862
  Phase 2 steps      : 11
  Val/Test steps     : 3

  NOTE: TTA transforms will be defined in Cell 8 after training
✓ Cell 3 complete


In [4]:
# ╔══════════════════════════════════════════════════════════╗
#  CELL 4 — Dataset Class + DataLoaders
#  NOTE: Building Phase 2 loaders directly — skipping Phase 1
#  since we are resuming from a saved checkpoint
# ╚══════════════════════════════════════════════════════════╝

# ── Dataset class ─────────────────────────────────────────────
class DeepfakeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = torch.tensor(row[LABEL_COL], dtype=torch.long)
        try:
            img = Image.open(row[PATH_COL]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (S, S), 0)
        if self.transform:
            img = self.transform(img)
        return img, label

# ── DataLoader factory ────────────────────────────────────────
def make_loaders(train_transform, batch_size):
    kw = dict(
        num_workers        = CFG['num_workers'],
        pin_memory         = True,
        prefetch_factor    = 2,
        persistent_workers = True,
    )
    train_ds = DeepfakeDataset(train_df, train_transform)
    val_ds   = DeepfakeDataset(val_df,   val_transform)
    test_ds  = DeepfakeDataset(test_df,  val_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  drop_last=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, drop_last=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, drop_last=False, **kw)
    return train_loader, val_loader, test_loader

# ── Build Phase 2 loaders directly ───────────────────────────
train_loader, val_loader, test_loader = make_loaders(
    phase2_transform, batch_size=CFG['phase2_batch']
)

# ── Verification ──────────────────────────────────────────────
print("=" * 60)
print("  DATALOADER VERIFICATION")
print("=" * 60)
print(f"  Train : {len(train_loader):,} batches  "
      f"({len(train_loader.dataset):,} images)")
print(f"  Val   : {len(val_loader):,} batches    "
      f"({len(val_loader.dataset):,} images)")
print(f"  Test  : {len(test_loader):,} batches   "
      f"({len(test_loader.dataset):,} images)")
print(f"  Batch size   : {CFG['phase2_batch']}")
print(f"  Eff. batch   : {CFG['phase2_batch'] * CFG['accum_steps']}")
print()

imgs, labels = next(iter(train_loader))
print(f"  Batch shape  : {tuple(imgs.shape)}")
print(f"  Labels sample: {labels[:8].tolist()}")
print(f"  Image dtype  : {imgs.dtype}")
print(f"  Image min/max: {imgs.min():.3f} / {imgs.max():.3f}")

n_real = (labels == 0).sum().item()
n_fake = (labels == 1).sum().item()
print(f"  Batch balance: {n_real} real | {n_fake} fake "
      f"(ideal {CFG['phase2_batch']//2}/{CFG['phase2_batch']//2})")

# ── Loader speed test ─────────────────────────────────────────
print()
print("  LOADER SPEED TEST (10 batches) ...")
import time as _time
t0 = _time.time()
for i, (imgs, labels) in enumerate(train_loader):
    if i == 9:
        break
elapsed = _time.time() - t0
print(f"  10 batches in {elapsed:.2f}s  "
      f"→ {10 * CFG['phase2_batch'] / elapsed:.0f} images/sec")

# ── Estimate time per epoch ───────────────────────────────────
secs_per_batch  = elapsed / 10
secs_per_epoch  = secs_per_batch * len(train_loader)
print()
print(f"  Est. time/epoch : ~{secs_per_epoch/60:.0f} min  "
      f"({len(train_loader):,} batches × {secs_per_batch:.2f}s)")
print(f"  Est. 13 epochs  : ~{13 * secs_per_epoch / 3600:.1f}h")
print(f"  Budget remaining: fits in 11.5h ✓" 
      if 13 * secs_per_epoch < 11.5 * 3600 
      else "  ⚠ May be tight — monitor time")

if torch.cuda.is_available():
    free, total_mem = torch.cuda.mem_get_info(0)
    print()
    print(f"  GPU memory used : {(total_mem-free)/1e9:.2f} GB / "
          f"{total_mem/1e9:.1f} GB total")

print()
print("=" * 60)
print("✓ Cell 4 complete")


  DATALOADER VERIFICATION
  Train : 7,494 batches  (239,828 images)
  Val   : 1,012 batches    (32,374 images)
  Test  : 500 batches   (15,982 images)
  Batch size   : 32
  Eff. batch   : 64

  Batch shape  : (32, 3, 224, 224)
  Labels sample: [0, 1, 0, 1, 1, 1, 0, 1]
  Image dtype  : torch.float32
  Image min/max: -2.118 / 2.640
  Batch balance: 16 real | 16 fake (ideal 16/16)

  LOADER SPEED TEST (10 batches) ...
  10 batches in 3.46s  → 92 images/sec

  Est. time/epoch : ~43 min  (7,494 batches × 0.35s)
  Est. 13 epochs  : ~9.4h
  Budget remaining: fits in 11.5h ✓

  GPU memory used : 0.11 GB / 15.6 GB total

✓ Cell 4 complete


In [5]:
# ╔══════════════════════════════════════════════════════════╗
#  CELL 5 — Model Definition + Load Checkpoint
#  - Same ConvNeXtV2Deepfake architecture as last session
#  - Load weights from Kaggle model input (epoch 6 checkpoint)
#  - Unfreeze backbone immediately — skip Phase 1 entirely
# ╚══════════════════════════════════════════════════════════╝

class ConvNeXtV2Deepfake(nn.Module):
    def __init__(self, model_name, dropout, drop_path_rate, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained     = False,
            num_classes    = 0,
            drop_rate      = dropout,
            drop_path_rate = drop_path_rate,
        )
        feat_dim = self.backbone.num_features  # 1024 for convnextv2_base

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
        print("  Backbone FROZEN — head only")

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
        print("  Backbone UNFROZEN — full fine-tuning enabled")

# ── Build model ───────────────────────────────────────────────
model = ConvNeXtV2Deepfake(
    model_name     = CFG['model_name'],
    dropout        = CFG['dropout'],
    drop_path_rate = CFG['drop_path_rate'],
    num_classes    = CFG['num_classes'],
).to(DEVICE)

# ── Load checkpoint from Kaggle model ────────────────────────
print("=" * 60)
print("  LOADING CHECKPOINT ...")
print("=" * 60)

state = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True)
missing, unexpected = model.load_state_dict(state, strict=True)
print(f"  ✓ Checkpoint loaded : {CKPT_PATH}")
print(f"  Missing keys        : {len(missing)}")
print(f"  Unexpected keys     : {len(unexpected)}")

# ── Unfreeze backbone — full fine-tune from first epoch ───────
model.unfreeze_backbone()

# ── Parameter counts ──────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad) / 1e6
backbone_params  = sum(p.numel() for p in model.backbone.parameters()) / 1e6
head_params      = sum(p.numel() for p in model.head.parameters()) / 1e6

print()
print("=" * 60)
print("  MODEL SUMMARY")
print("=" * 60)
print(f"  Backbone        : {CFG['model_name']}")
print(f"  Feature dim     : {model.backbone.num_features}")
print(f"  Total params    : {total_params:.3f}M")
print(f"  Backbone params : {backbone_params:.3f}M")
print(f"  Head params     : {head_params:.3f}M")
print(f"  Trainable now   : {trainable_params:.3f}M  (all unfrozen ✓)")

# ── Dry-run forward pass ──────────────────────────────────────
print()
print("  DRY-RUN FORWARD PASS ...")
model.eval()
with torch.no_grad():
    dummy = torch.randn(4, 3, 224, 224, device=DEVICE)
    out   = model(dummy)
    probs = F.softmax(out, dim=1)
print(f"  Input  : {tuple(dummy.shape)}")
print(f"  Output : {tuple(out.shape)}  ← (batch=4, classes=2) ✓")
print(f"  Probs sum/sample : "
      f"{probs.sum(dim=1).cpu().numpy().round(4)}  ← must all be 1.0")

# ── GPU memory after model load ───────────────────────────────
free, total_mem = torch.cuda.mem_get_info(0)
print()
print(f"  GPU memory used : {(total_mem-free)/1e9:.2f} GB / "
      f"{total_mem/1e9:.1f} GB total")
print(f"  GPU memory free : {free/1e9:.2f} GB")

model.train()
print()
print("=" * 60)
print("✓ Cell 5 complete — ready for Phase 2 training")


  LOADING CHECKPOINT ...
  ✓ Checkpoint loaded : /kaggle/input/models/omkarphadtare30/convnextv2/other/default/1/best_convnextv2.pth
  Missing keys        : 0
  Unexpected keys     : 0
  Backbone UNFROZEN — full fine-tuning enabled

  MODEL SUMMARY
  Backbone        : convnextv2_base
  Feature dim     : 1024
  Total params    : 88.221M
  Backbone params : 87.693M
  Head params     : 0.528M
  Trainable now   : 88.221M  (all unfrozen ✓)

  DRY-RUN FORWARD PASS ...
  Input  : (4, 3, 224, 224)
  Output : (4, 2)  ← (batch=4, classes=2) ✓
  Probs sum/sample : [1. 1. 1. 1.]  ← must all be 1.0

  GPU memory used : 0.90 GB / 15.6 GB total
  GPU memory free : 14.74 GB

✓ Cell 5 complete — ready for Phase 2 training


In [6]:
# ╔══════════════════════════════════════════════════════════╗
#  CELL 6 — MixUp / CutMix + Training Loop
#  Identical to last session — no changes needed
# ╚══════════════════════════════════════════════════════════╝

# ── MixUp ─────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    mixed = lam * x + (1 - lam) * x[index]
    return mixed, y, y[index], lam

# ── CutMix ────────────────────────────────────────────────────
def cutmix_data(x, y, alpha=0.4):
    lam        = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    B, _, H, W = x.shape
    index      = torch.randperm(B, device=x.device)
    cut_rat    = np.sqrt(1.0 - lam)
    cut_w      = int(W * cut_rat)
    cut_h      = int(H * cut_rat)
    cx         = np.random.randint(W)
    cy         = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return mixed, y, y[index], lam

def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ── Training loop ─────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer, scaler,
              is_train=True, use_mix=False, accum_steps=1):
    model.train() if is_train else model.eval()

    total_loss = 0.0
    correct    = 0
    total      = 0
    all_probs, all_preds, all_labels = [], [], []

    if is_train:
        optimizer.zero_grad()

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for i, (imgs, labels) in enumerate(tqdm(
                loader,
                desc='Train' if is_train else 'Val  ',
                leave=False)):

            imgs   = imgs.to(DEVICE,   non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            # MixUp / CutMix
            use_aug = is_train and use_mix and (
                random.random() < CFG['mix_prob'])
            if use_aug:
                if random.random() < 0.5:
                    imgs, y_a, y_b, lam = mixup_data(
                        imgs, labels, CFG['mixup_alpha'])
                else:
                    imgs, y_a, y_b, lam = cutmix_data(
                        imgs, labels, CFG['cutmix_alpha'])

            # Forward
            with autocast('cuda'):
                logits = model(imgs)
                if use_aug:
                    loss = mixed_criterion(criterion, logits, y_a, y_b, lam)
                else:
                    loss = criterion(logits, labels)
                loss = loss / accum_steps

            # Backward
            if is_train:
                scaler.scale(loss).backward()
                if (i + 1) % accum_steps == 0 or (i + 1) == len(loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), CFG['grad_clip'])
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            # Metrics
            with torch.no_grad():
                probs = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
                preds = logits.argmax(dim=1).detach().cpu().numpy()
                labs  = labels.detach().cpu().numpy()

            total_loss += loss.item() * accum_steps * labels.size(0)
            correct    += (preds == labs).sum()
            total      += labels.size(0)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labs)

    avg_loss = total_loss / total
    accuracy = correct / total
    auc      = roc_auc_score(all_labels, all_probs) \
               if len(set(all_labels)) > 1 else 0.0
    return avg_loss, accuracy, auc, all_preds, all_labels

# ── Quick sanity check ────────────────────────────────────────
print("=" * 60)
print("  MIXUP / CUTMIX / TRAINING LOOP VERIFICATION")
print("=" * 60)
_imgs, _labels = next(iter(train_loader))
_imgs, _labels = _imgs.to(DEVICE), _labels.to(DEVICE)

mx_imgs, ya, yb, lam_mx = mixup_data(_imgs, _labels, CFG['mixup_alpha'])
cm_imgs, ya, yb, lam_cm = cutmix_data(_imgs, _labels, CFG['cutmix_alpha'])

_crit      = nn.CrossEntropyLoss()
_dummy     = torch.randn(_imgs.size(0), 2, device=DEVICE)
_loss_test = mixed_criterion(_crit, _dummy, ya, yb, lam_cm)

print(f"  MixUp  λ={lam_mx:.4f}  shape={tuple(mx_imgs.shape)}  ✓")
print(f"  CutMix λ={lam_cm:.4f}  shape={tuple(cm_imgs.shape)}  ✓")
print(f"  mixed_criterion loss : {_loss_test.item():.4f}  ✓")

del _imgs, _labels, mx_imgs, cm_imgs, _dummy, _crit
gc.collect()
torch.cuda.empty_cache()

free, total_mem = torch.cuda.mem_get_info(0)
print(f"  GPU free after cleanup : {free/1e9:.2f} GB")
print()
print("=" * 60)
print("✓ Cell 6 complete — ready to launch Phase 2 training")


  MIXUP / CUTMIX / TRAINING LOOP VERIFICATION
  MixUp  λ=0.0888  shape=(32, 3, 224, 224)  ✓
  CutMix λ=0.6314  shape=(32, 3, 224, 224)  ✓
  mixed_criterion loss : 0.8457  ✓
  GPU free after cleanup : 14.76 GB

✓ Cell 6 complete — ready to launch Phase 2 training


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
#  CELL 7 — Phase 2 Continued Training (epochs 7–19)
#  Resuming from Val Acc 93.98% | AUC 0.9848 (epoch 6)
#  Full backbone fine-tune + strong aug + MixUp/CutMix
# ╚══════════════════════════════════════════════════════════╝

# ── Criterion ─────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(
    weight          = class_weights,
    label_smoothing = CFG['label_smoothing'],
)

# ── Discriminative LRs — same as last session ─────────────────
backbone_params = list(model.backbone.parameters())
head_params     = list(model.head.parameters())

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CFG['phase2_lr_backbone']},
    {'params': head_params,     'lr': CFG['phase2_lr_head']},
], weight_decay=CFG['weight_decay'])

# ── ReduceLROnPlateau on val accuracy ─────────────────────────
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode       = 'max',
    factor     = CFG['rlrop_factor'],
    patience   = CFG['rlrop_patience'],
    min_lr     = CFG['rlrop_min_lr'],
)

scaler = GradScaler('cuda')

# ── History ───────────────────────────────────────────────────
history = dict(
    train_loss=[], val_loss=[],
    train_acc=[],  val_acc=[],
    val_auc=[],
)

# ── Start from best known baseline ────────────────────────────
best_val_acc = 0.9398   # 93.98% from epoch 6
best_val_auc = 0.9848
patience_ctr = 0

print("=" * 65)
print("  PHASE 2 RESUMED — Full Fine-Tune")
print("=" * 65)
print(f"  Baseline (epoch 6) : Val Acc 93.98%  AUC 0.9848")
print(f"  Epochs to run      : {CFG['phase2_epochs']}  "
      f"(patience={CFG['phase2_patience']})")
print(f"  Backbone LR        : {CFG['phase2_lr_backbone']}")
print(f"  Head LR            : {CFG['phase2_lr_head']}")
print(f"  Eff. batch size    : {CFG['phase2_batch'] * CFG['accum_steps']}")
print(f"  Time budget        : {CFG['time_budget_sec']/3600:.1f}h")
print("=" * 65)

for epoch in range(1, CFG['phase2_epochs'] + 1):

    # ── Time budget guard ─────────────────────────────────────
    elapsed   = time.time() - TRAIN_START
    remaining = CFG['time_budget_sec'] - elapsed
    print(f"\n  [Elapsed {elapsed/3600:.2f}h | Remaining {remaining/3600:.2f}h]")

    if remaining < 3600:
        print("  ⚠ Less than 1h remaining — stopping to preserve time for testing")
        break

    # ── Train ─────────────────────────────────────────────────
    tr_loss, tr_acc, tr_auc, _, _ = run_epoch(
        model, train_loader, criterion, optimizer, scaler,
        is_train=True, use_mix=True, accum_steps=CFG['accum_steps'],
    )

    # ── Validate ──────────────────────────────────────────────
    va_loss, va_acc, va_auc, _, _ = run_epoch(
        model, val_loader, criterion, optimizer, scaler,
        is_train=False,
    )

    # ── Scheduler step on val accuracy ────────────────────────
    scheduler.step(va_acc)

    # ── Record history ────────────────────────────────────────
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    history['val_auc'].append(va_auc)

    # ── LR readout ────────────────────────────────────────────
    bb_lr = optimizer.param_groups[0]['lr']
    hd_lr = optimizer.param_groups[1]['lr']

    # ── Save best ─────────────────────────────────────────────
    saved = ''
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_val_auc = va_auc
        torch.save(model.state_dict(), CFG['model_path'])
        saved        = '  ← saved ✓'
        patience_ctr = 0
    else:
        patience_ctr += 1

    print(f"  Ep {epoch:02d}/{CFG['phase2_epochs']:02d} | "
          f"Train Loss {tr_loss:.4f}  Acc {tr_acc:.4f} | "
          f"Val Loss {va_loss:.4f}  Acc {va_acc:.4f}  "
          f"AUC {va_auc:.4f} | "
          f"LR bb={bb_lr:.1e} hd={hd_lr:.1e}"
          f"{saved}")

    # ── Early stopping ────────────────────────────────────────
    if patience_ctr >= CFG['phase2_patience']:
        print(f"\n  Early stopping at epoch {epoch} "
              f"(no improvement for {CFG['phase2_patience']} epochs)")
        break

# ── Final summary ─────────────────────────────────────────────
elapsed = time.time() - TRAIN_START
print()
print("=" * 65)
print(f"  TRAINING COMPLETE")
print("=" * 65)
print(f"  Best Val Acc : {best_val_acc*100:.2f}%")
print(f"  Best Val AUC : {best_val_auc:.4f}")
print(f"  Checkpoint   : {CFG['model_path']}")
print(f"  Total elapsed: {elapsed/3600:.2f}h")
print("=" * 65)
print("✓ Cell 7 complete — proceed to Cell 8 for testing")


  PHASE 2 RESUMED — Full Fine-Tune
  Baseline (epoch 6) : Val Acc 93.98%  AUC 0.9848
  Epochs to run      : 15  (patience=8)
  Backbone LR        : 5e-06
  Head LR            : 5e-05
  Eff. batch size    : 64
  Time budget        : 11.5h

  [Elapsed 0.08h | Remaining 11.42h]


  Ep 01/15 | Train Loss 0.2984  Acc 0.8660 | Val Loss 0.3478  Acc 0.9227  AUC 0.9807 | LR bb=5.0e-06 hd=5.0e-05

  [Elapsed 1.80h | Remaining 9.70h]


  Ep 02/15 | Train Loss 0.2881  Acc 0.8735 | Val Loss 0.3100  Acc 0.9332  AUC 0.9802 | LR bb=5.0e-06 hd=5.0e-05

  [Elapsed 3.51h | Remaining 7.99h]


  Ep 03/15 | Train Loss 0.2876  Acc 0.8727 | Val Loss 0.3403  Acc 0.9267  AUC 0.9837 | LR bb=5.0e-06 hd=5.0e-05

  [Elapsed 5.22h | Remaining 6.28h]


  Ep 04/15 | Train Loss 0.2811  Acc 0.8752 | Val Loss 0.2780  Acc 0.9473  AUC 0.9845 | LR bb=5.0e-06 hd=5.0e-05  ← saved ✓

  [Elapsed 6.94h | Remaining 4.56h]


  Ep 05/15 | Train Loss 0.2782  Acc 0.8754 | Val Loss 0.2893  Acc 0.9428  AUC 0.9840 | LR bb=5.0e-06 hd=5.0e-05

  [Elapsed 8.65h | Remaining 2.85h]


  Ep 06/15 | Train Loss 0.2734  Acc 0.8808 | Val Loss 0.2854  Acc 0.9443  AUC 0.9839 | LR bb=5.0e-06 hd=5.0e-05

  [Elapsed 10.36h | Remaining 1.14h]


KeyboardInterrupt: 

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
#  CELL 8 — Full Test Evaluation
#  Matches exact structure from convnextv2-testing notebook
#  Part A : Simple test (no TTA)
#  Part B : TTA (4 views) using tta_predict()
#  Part C : Threshold tuning
#  Part D : Plots + Save CSV
# ╚══════════════════════════════════════════════════════════╝

# ── Load best checkpoint ──────────────────────────────────────
print("=" * 65)
print("  LOADING BEST CHECKPOINT FOR TESTING ...")
print("=" * 65)
state = torch.load(CFG['model_path'], map_location=DEVICE,
                   weights_only=True)
model.load_state_dict(state, strict=True)
model.eval()
print(f"  ✓ Loaded : {CFG['model_path']}")
print(f"  Best Val Acc : {best_val_acc*100:.2f}%")
print(f"  Best Val AUC : {best_val_auc:.4f}")

# ── Build clean test loader (val_transform, batch=64) ─────────
kw = dict(num_workers=CFG['num_workers'], pin_memory=True,
          prefetch_factor=2, persistent_workers=True)
test_ds_clean  = DeepfakeDataset(test_df, val_transform)
test_loader_64 = DataLoader(test_ds_clean, batch_size=64,
                             shuffle=False, drop_last=False, **kw)

# ════════════════════════════════════════════════════════════
#  PART A — Simple Test (no TTA)
# ════════════════════════════════════════════════════════════
print()
print("=" * 65)
print("  PART A — SIMPLE TEST (no TTA)")
print("=" * 65)

simple_probs, simple_preds, simple_labels = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader_64, desc="Simple Test"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast("cuda"):
            logits = model(imgs)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        simple_probs.extend(probs)
        simple_preds.extend(preds)
        simple_labels.extend(labels.numpy())

simple_probs  = np.array(simple_probs)
simple_preds  = np.array(simple_preds)
simple_labels = np.array(simple_labels)

simple_acc = (simple_preds == simple_labels).mean()
simple_auc = roc_auc_score(simple_labels, simple_probs)

print(f"  Simple Test Accuracy : {simple_acc*100:.2f}%")
print(f"  Simple Test AUC      : {simple_auc:.4f}")
print()
print(classification_report(simple_labels, simple_preds,
                             target_names=["Real", "Fake"], digits=4))

# ════════════════════════════════════════════════════════════
#  PART B — TTA (4 views) — exact tta_predict() from testing nb
# ════════════════════════════════════════════════════════════
print("=" * 65)
print("  PART B — TTA EVALUATION (4 views)")
print("=" * 65)

S    = CFG['img_size']
MEAN = CFG['img_mean']
STD  = CFG['img_std']

TTA_TRANSFORMS = [
    T.Compose([T.Resize((S, S)),
               T.ToTensor(), T.Normalize(MEAN, STD)]),
    T.Compose([T.Resize((S, S)),
               T.RandomHorizontalFlip(p=1.0),
               T.ToTensor(), T.Normalize(MEAN, STD)]),
    T.Compose([T.Resize((S+32, S+32)),
               T.CenterCrop(S),
               T.ToTensor(), T.Normalize(MEAN, STD)]),
    T.Compose([T.Resize((S, S)),
               T.ColorJitter(brightness=0.15, contrast=0.15),
               T.ToTensor(), T.Normalize(MEAN, STD)]),
]

def tta_predict(model, df, tta_transforms):
    model.eval()
    all_avg_probs = None
    all_labels    = None

    for t_idx, transform in enumerate(tta_transforms):
        ds     = DeepfakeDataset(df, transform)
        loader = DataLoader(ds, batch_size=64, shuffle=False,
                            num_workers=CFG['num_workers'],
                            pin_memory=True,
                            persistent_workers=True)
        probs_list, labels_list = [], []
        with torch.no_grad():
            for imgs, labels in tqdm(loader,
                                     desc=f"TTA {t_idx+1}/{len(tta_transforms)}",
                                     leave=False):
                imgs = imgs.to(DEVICE, non_blocking=True)
                with autocast("cuda"):
                    logits = model(imgs)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                probs_list.append(probs)
                if t_idx == 0:
                    labels_list.extend(labels.numpy())

        probs_arr = np.concatenate(probs_list, axis=0)
        if all_avg_probs is None:
            all_avg_probs = probs_arr
            all_labels    = np.array(labels_list)
        else:
            all_avg_probs += probs_arr

        print(f"  View {t_idx+1} done — "
              f"mean fake prob: {probs_arr[:, 1].mean():.4f}")

    all_avg_probs /= len(tta_transforms)
    return (np.argmax(all_avg_probs, axis=1),
            all_labels,
            all_avg_probs[:, 1])

print(f"  TTA views : {len(TTA_TRANSFORMS)}")
tta_preds, tta_labels, tta_probs = tta_predict(
    model, test_df, TTA_TRANSFORMS)

tta_acc = (tta_preds == tta_labels).mean()
tta_auc = roc_auc_score(tta_labels, tta_probs)

print()
print("=" * 65)
print("  TTA TEST RESULTS — ConvNeXt V2 (resumed)")
print("=" * 65)
print(f"  TTA Accuracy : {tta_acc*100:.2f}%")
print(f"  TTA AUC      : {tta_auc:.4f}")
print()
print(classification_report(tta_labels, tta_preds,
                             target_names=["Real", "Fake"], digits=4))

# ════════════════════════════════════════════════════════════
#  PART C — Threshold Tuning
# ════════════════════════════════════════════════════════════
print("=" * 65)
print("  PART C — THRESHOLD TUNING")
print("=" * 65)

thresholds   = np.arange(0.30, 0.71, 0.01)
best_thresh  = 0.50
best_acc     = tta_acc
best_f1      = f1_score(tta_labels, tta_preds)

for t in thresholds:
    preds_t = (tta_probs >= t).astype(int)
    acc_t   = (preds_t == tta_labels).mean()
    f1_t    = f1_score(tta_labels, preds_t)
    if acc_t > best_acc:
        best_acc    = acc_t
        best_thresh = t
        best_f1     = f1_t

tta_preds_best = (tta_probs >= best_thresh).astype(int)
tta_acc_best   = (tta_preds_best == tta_labels).mean()

print(f"  Best threshold : {best_thresh:.2f}")
print(f"  Best TTA Acc   : {tta_acc_best*100:.2f}%")
print(f"  Best F1        : {best_f1:.4f}")
print()
print("  Final Classification Report (best threshold):")
print(classification_report(tta_labels, tta_preds_best,
                             target_names=["Real", "Fake"], digits=4))

# ════════════════════════════════════════════════════════════
#  PART D — Plots + Save CSV
# ════════════════════════════════════════════════════════════
print("=" * 65)
print("  PART D — PLOTS + SAVING ...")
print("=" * 65)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    f'ConvNeXt V2 v3 — Test Results\n'
    f'Simple: {simple_acc*100:.2f}%  |  '
    f'TTA: {tta_acc_best*100:.2f}%  |  '
    f'AUC: {tta_auc:.4f}',
    fontsize=14, fontweight='bold'
)

# Confusion matrix — simple
cm = confusion_matrix(simple_labels, simple_preds)
ConfusionMatrixDisplay(cm, display_labels=["Real","Fake"]).plot(
    ax=axes[0,0], colorbar=False, cmap="Blues")
axes[0,0].set_title("Confusion Matrix — Simple", fontweight='bold')

# Confusion matrix — TTA best threshold
cm2 = confusion_matrix(tta_labels, tta_preds_best)
ConfusionMatrixDisplay(cm2, display_labels=["Real","Fake"]).plot(
    ax=axes[0,1], colorbar=False, cmap="Greens")
axes[0,1].set_title(f"Confusion Matrix — TTA (t={best_thresh:.2f})",
                    fontweight='bold')

# ROC curve
fpr,  tpr,  _ = roc_curve(tta_labels,    tta_probs)
fpr_s,tpr_s,_ = roc_curve(simple_labels, simple_probs)
axes[0,2].plot(fpr,   tpr,   color='darkorange', lw=2,
               label=f'TTA    AUC={tta_auc:.4f}')
axes[0,2].plot(fpr_s, tpr_s, color='steelblue',  lw=2, ls='--',
               label=f'Simple AUC={simple_auc:.4f}')
axes[0,2].plot([0,1],[0,1],'k--',lw=1)
axes[0,2].set_xlabel('FPR'); axes[0,2].set_ylabel('TPR')
axes[0,2].set_title('ROC Curve', fontweight='bold')
axes[0,2].legend(loc='lower right'); axes[0,2].grid(alpha=0.3)

# Threshold vs accuracy
t_vals   = np.arange(0.30, 0.71, 0.01)
acc_vals = [(( tta_probs >= t).astype(int) == tta_labels).mean()*100
             for t in t_vals]
axes[1,0].plot(t_vals, acc_vals, color='purple', lw=2)
axes[1,0].axvline(best_thresh, color='red', ls='--',
                  label=f'Best={best_thresh:.2f}')
axes[1,0].set_xlabel('Threshold'); axes[1,0].set_ylabel('Accuracy (%)')
axes[1,0].set_title('Threshold vs Accuracy', fontweight='bold')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# Training history this session
epochs_x = list(range(7, 7 + len(history['val_acc'])))
axes[1,1].plot(epochs_x, [v*100 for v in history['train_acc']],
               label='Train Acc', color='steelblue', lw=2)
axes[1,1].plot(epochs_x, [v*100 for v in history['val_acc']],
               label='Val Acc', color='darkorange', lw=2)
axes[1,1].axhline(93.98, color='gray', ls='--', lw=1,
                  label='Baseline 93.98%')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].set_title('Training History (session 2)',fontweight='bold')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

# Probability distribution
real_p = tta_probs[tta_labels == 0]
fake_p = tta_probs[tta_labels == 1]
axes[1,2].hist(real_p, bins=50, alpha=0.6, color='steelblue',
               label=f'Real (n={len(real_p):,})')
axes[1,2].hist(fake_p, bins=50, alpha=0.6, color='tomato',
               label=f'Fake (n={len(fake_p):,})')
axes[1,2].axvline(best_thresh, color='black', ls='--',
                  label=f'Threshold={best_thresh:.2f}')
axes[1,2].set_xlabel('Predicted Prob (Fake)')
axes[1,2].set_ylabel('Count')
axes[1,2].set_title('Probability Distribution', fontweight='bold')
axes[1,2].legend(); axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/convnextv2_v3_test_results.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("  ✓ Saved convnextv2_v3_test_results.png")

# Save CSV
pd.DataFrame({
    'imagepath'     : test_df[PATH_COL].values,
    'true_label'    : tta_labels,
    'simple_pred'   : simple_preds,
    'simple_prob'   : simple_probs.round(4),
    'tta_prob'      : tta_probs.round(4),
    'tta_pred_05'   : tta_preds,
    'tta_pred_best' : tta_preds_best,
    'correct_simple': (simple_preds    == simple_labels).astype(int),
    'correct_tta'   : (tta_preds_best  == tta_labels).astype(int),
}).to_csv('/kaggle/working/convnextv2_v3_predictions.csv', index=False)
print("  ✓ Saved convnextv2_v3_predictions.csv")

print()
print("=" * 65)
print("  FINAL RESULTS SUMMARY")
print("=" * 65)
print(f"  Simple Test Acc : {simple_acc*100:.2f}%  |  AUC: {simple_auc:.4f}")
print(f"  TTA Acc (0.50)  : {tta_acc*100:.2f}%  |  AUC: {tta_auc:.4f}")
print(f"  TTA Acc (best)  : {tta_acc_best*100:.2f}%  [threshold={best_thresh:.2f}]")
print(f"  vs baseline     : {(tta_acc_best - 0.9432)*100:+.2f}%  (was 94.32%)")
print("=" * 65)
print("✓ Cell 8 complete — ConvNeXt V2 v3 done!")
